#### Volumes 

- #####Imagine we have files on Azure storage location / on-prem DB
1. so we can either import them as tables
2.  or we can just create volume in DBR then upload those files as is (csv,excel) to the volume then process them as files and write back 
3. So like Schemas, we create volumes in DBR (raw_data under worskpace schema in this case) & uploaded order.csv file there.

#### SHOW vs Display

- Both are Actions 
- Show - Spark specific / Display DBR specific 
- SHow is bit lightweight compared to display 
- Display provides sorting pagination etc unlike show
- df.show(n) Vs display(df.limit(n))

#### Reading files from volume 

### Read csv , parquets 
- spark.read.csv
- spark.read.parquet
- spark.read.table

### Write csv , paruets
- df.write.format('csv').option().save("<PATH>")
- df.write.format('parquet').options().save("<PATH>")

In [0]:
df_csv_file = spark.read \
    .option("Header", True) \
    .option("InferSchema", True) \
    .csv("/Volumes/workspace/default/raw_data/orders.csv")
"""
spark.read → Creates a DataFrameReader object
.option() → Just sets configuration metadata
.csv() → Defines the logical plan to read CSV
thats why all configuration options like option / infer schema etc must be given after spark dataframe reader . 
"""
# display(df_csv_file)
# df_csv_file.show()
df_csv_file.printSchema()

### Applying manually given schema 


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
manual_schema = StructType([
    StructField('order_date', DateType(), True),
    StructField('country', StringType(), True),
    StructField('order_id', IntegerType()),
    StructField('product', StringType()),
    StructField('qty', IntegerType(), True),
    StructField('price', DoubleType())
])

CSV_PATH = "/Volumes/workspace/default/raw_data/orders.csv"

# 2 methods to read csv 
## spark.read.csv OR spark.read.format('csv')
manual_schm_csv = spark.read \
    .option("Header" ,True) \
    .schema(manual_schema) \
    .option("DateFormat", "yyyy-MM-dd") \
    .csv(CSV_PATH)

manual_schm_csv.show(5)
display(manual_schm_csv.limit(3))

### Lets add columns & write format as Parquet (default format of DBR)


In [0]:
from pyspark.sql.functions import col
manual_schm_csv = manual_schm_csv.withColumn('total_order_cost',col('qty') * col('price'))
manual_schm_csv.show(1)

DEST_PATH = "/Volumes/workspace/default/raw_data/orders"
manual_schm_csv.write \
    .format('parquet') \
    .mode('overwrite') \
    .save(DEST_PATH)

##### How to list files in volume rather than going there manually for checking 

- %fs
ls /Volumes/workspace/default/raw_data/orders
##### OR

- dbutils.fs.ls("/Volumes/workspace/default/raw_data/orders") 

In [0]:
# %fs
# ls /Volumes/workspace/default/raw_data/orders

display(dbutils.fs.ls(DEST_PATH))
